<a href="https://colab.research.google.com/github/Creater2036/iGEM/blob/main/Vina_Docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMPORTANT

This notebook has 2 parts to it, one of which is checking for necessary residues between the original protein and the new one, and the second part which is running autodock Vina for checking affinity of the new protein.

For PART 1, you need:
- The pdb of the original protein
- The pdb of the newly generated protein
- The contigs of the newly generated protein that were chosen (insert below in Aligning Proteins section)

Note: If you are doing something other than beta lactamase, you would want to change the orig_aligns variable as it check for necessary residues in beta lactamase, not other proteins.

For PART 2, you need:
- The pdb of newly generated protein
- The sdf (ligand file) of the protein

To upload these files, go to the left bar and click on the folder. Click the upload file button and upload the necessary files

NOTE: After running this cell below, your colab WILL crash. This is normal, just continue running the rest of the cells as usual

In [ ]:
# @title (1) Install Condacolab (< 1min)
%%time

! pip install -q condacolab
import condacolab
condacolab.install()

#@markdown > An automatic restart of the kernel is expected after the execution of this block.
#@markdown >
#@markdown > Stay connected to the same runtime and proceed to the next code block!


⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:07
🔁 Restarting kernel...
CPU times: user 338 ms, sys: 153 ms, total: 491 ms
Wall time: 9.2 s


# File Names

Input the names of all of the files you plan on using here and make sure to upload the file using the instructions above, if you're not planning on uploading all of them, leave the field blank. **ADD THE .pdb or .sdf part at the end**

In [ ]:
#@markdown # Part 1
orig_protein = "1btl.pdb" #@param {type:"string"}
#@markdown Insert original protein pdb file here
new_protein = "best_design0.pdb" #@param {type:"string"}
#@markdown Insert novel protein pdb file here
#[70, 73, 130, 132, 166, 234, 237]
orig_aligns =  [70, 73] #@param
#@markdown - Change this orig_aligns if you want to compare different residues than the ones given here (make sure to put it in list format like shown)
resid_format = "4-4/A68-84/26-26/A125-238/14-14" #@param {type: "string"}
#@markdown - This is where you put the contigs of your new protein (This should be inside the csv file)

#@markdown ---
#@markdown # Part 2
sdf_protein = "PNM_ideal.sdf" #@param {type:"string"}
#@markdown - Insert sdf of original protein here


# Part 1

In [ ]:
!conda install -c conda-forge -c schrodinger pymol-bundle -q

Channels:
 - conda-forge
 - schrodinger
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - pymol-bundle


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    alsa-lib-1.2.13            |       hb9d3cd8_0         547 KB  conda-forge
    apbs-1.5                   |       hd8ae8d1_0        22.8 MB  conda-forge
    attr-2.5.1                 |       h166bdaf_1          69 KB  conda-forge
    biopython-1.85             |  py311h9ecbd09_1         3.3 MB  conda-forge
    blosc-1.21.6               |       he440d0b_1          47 KB  conda-forge
    ca-certificates-2025.1.31  |       hbcca054_0         154 KB  conda-forge
    cached-property-1.5.2      |       hd8ed1ab_1           4 KB  conda-forge
    cached_property-1.5.2      |     pyha770c72_1          11 KB  conda-forge
    cairo-1.18.2              

In [ ]:
def find_alignment(contigs, alignment_inds):
    locations = alignment_inds.copy()
    new_locs = []
    contigs = contigs.split("/")
    A_region = [i[1:].split("-") for i in contigs if "A" in i]
    added_region = [int(i.split("-")[0]) for i in contigs if "A" not in i]

    while locations:
        for i in range(len(A_region)):
            region = A_region[i]
            if int(region[0]) <= locations[0] < int(region[1]):  # Check if the location falls in the current region
                loc = (locations[0] - int(region[0]) +
                       sum(added_region[0:i+1]) +  # Sum of added regions before this one
                       sum([int(x[1]) - int(x[0]) for x in A_region[:i]]) +  # Sum of A region lengths before this one
                       i + 1)  # Add one for each inserted region
                # print(locations[0])
                # print(region[0])
                # print(sum(added_region[0:i+1]))
                # print(sum([int(x[1]) - int(x[0]) for x in A_region[:i]]))
                # print(i)
                new_locs.append(loc)
                locations.pop(0)  # Remove the processed location
                break  # Stop processing this location
    return new_locs

new_aligns = find_alignment(resid_format, orig_aligns)
new_aligns

[7, 10]

Do not rerun this line without disconnecting and deleting runtime, else it will put the images over each other and keep appending more to the txt's. This line takes a while to run

In [ ]:
import pymol
from pymol import cmd
import os

# Initialize PyMOL
cmd.reinitialize()

# Load the target and mobile structures
cmd.load(f"{orig_protein}", "target")  # Target structure
cmd.load(f"{new_protein}", "mobile")  # Mobile structure

# Check if both objects are loaded
print("Target Atom Count:", cmd.count_atoms("target"))
print("Mobile Atom Count:", cmd.count_atoms("mobile"))

# Align mobile structure onto the target
cmd.align("target", "mobile")

# Select Serine 130
cmd.select("Ser130_target", "resi 234 and target") #130
#cmd.select("within_6A", "Ser130_target around 6")

# Select corresponding Serine in mobile
cmd.select("Ser53_mobile", "resi 157 and mobile") #53

# Measure distances between the selected groups
#cmd.distance("130_gap", "mobile_within_6A", "within_6A")
#print(cmd.get_model("Ser53_mobile").atom)

#cmd.zoom()
#cmd.png(f'test.png', 500, 500, dpi = 300, ray = 1)
os.system('mkdir /content/Residues')

t = 0
for i in range(len(orig_aligns)):
  os.system(f'mkdir /content/Residues/Orig_{orig_aligns[i]}')
  print(f'For Original Residue {orig_aligns[i]} and New Residue {new_aligns[i]}')
  file_path = f"/content/Residues/Orig_{orig_aligns[i]}/distances.txt"

  orig_res = f"target and resi {orig_aligns[i]}"
  new_res = f"mobile and resi {new_aligns[i]}"
  distance = cmd.distance("dist", f"{orig_res}", f"{new_res}")
  with open(file_path, "a") as f:
    f.write(f"Distance between alpha carbons at residue {orig_aligns[i]} in original and residue {new_aligns[i]} in novel protein: {distance} Å\n\n")

  for atom1 in cmd.get_model("Ser130_target").atom:
      for atom2 in cmd.get_model("Ser53_mobile").atom:

        if atom1.name == atom2.name:
          atom1_selection = f"target and resi {orig_aligns[i]} and name {atom1.name}" #130
          atom2_selection = f"mobile and resi {new_aligns[i]} and name {atom2.name}" #53

          cmd.delete("focus_atoms")
          cmd.delete("distance_lines")
          #cmd.hide("everything")

          #dist = cmd.distance(f"dist_{atom1.index}_{atom2.index}", f"Ser130_target and id {atom1.index}", f"Ser53_mobile and id {atom2.index}")
          #cmd.select("focus_atoms", f"(id {atom1.index} and Ser130_target) or (id {atom2.index} and Ser53_mobile)")
          dist = cmd.distance("distance_lines", atom1_selection, atom2_selection)
          cmd.select("focus_atoms", f"({atom1_selection}) or ({atom2_selection})")
          cmd.zoom("focus_atoms", buffer = 5)  # Zoom in on the selected atoms

          # Represent the atoms and distance clearly
          cmd.show("sticks", "focus_atoms")
          cmd.label("focus_atoms", "name")
          cmd.show("spheres", "focus_atoms")
          cmd.set("sphere_scale", 0.3)
          cmd.color("yellow", f"id {atom1_selection}")  # Highlight atom in target
          cmd.color("cyan", f"id {atom2_selection}")
          cmd.color("magenta", "distance_lines")  # Highlight bond in magenta

          cmd.rotate("x", 30)  # Rotate around the x-axis by 30 degrees
          cmd.rotate("y", 20)  # Rotate around the y-axis by 20 degrees
          cmd.rotate("z", 45)
          cmd.translate([1.7, 0, 0], "focus_atoms")

          # Save a high-quality image of the view
          cmd.png(f"/content/Residues/Orig_{orig_aligns[i]}/distance_{atom1.name}_{atom2.name}.png", width=1000, height=800, dpi=300, ray=1)
          with open(file_path, "a") as f:
            f.write(f"Distance between {atom1.name} in original and {atom2.name} in new is {dist}\n")
          #print(f"Distance between {atom1.name} in original and {atom2.name} in new is {dist}")
          t+=1
  #print()


# Optional: Save session for further inspection
#cmd.save("session.pse")


Target Atom Count: 2236
Mobile Atom Count: 1311
For Original Residue 70 and New Residue 7


KeyboardInterrupt: 

# Part 2

In [ ]:
# @title (2) Install Packages and Data (~ 5min)
%%time

# Install Reduce2 (cctbx-base)
! conda install cctbx-base

!conda install openbabel


# Install Prody and py3Dmol
! pip install prody py3Dmol


# Download Phenix Project geostd (restraint) Library
goestd_repo = "https://github.com/phenix-project/geostd.git"
! git clone {goestd_repo}


# Install Meeko (develop branch) and Dependencies
! conda install numpy scipy rdkit
! git clone --single-branch --branch develop https://github.com/forlilab/Meeko.git
! cd Meeko; pip install --use-pep517 -e .; cd ..


# Install Scrubber (develop branch)
! git clone --single-branch --branch develop https://github.com/forlilab/scrubber.git
! cd scrubber; pip install --use-pep517 -e .; cd ..


# Download Vina Executables
! wget https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64
! mv vina_1.2.5_linux_x86_64 vina; chmod +x vina;

#@markdown > List of Python packages installed:
#@markdown > - (conda) cctbx-base numpy scipy rdkit
#@markdown > - (pip) prody py3Dmol
#@markdown > - (pip) (develop branch on GitHub) meeko scrubber

#@markdown > Executable and data downloaded:
#@markdown > - (release on GitHub) ccsb-scripps/AutoDock-Vina: vina_1.2.5_linux_x86_64
#@markdown > - (repository on GitHub) Phenix-project/geostd

Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - \ done

# All requested packages already installed.

Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / - done

# All requested packages already installed.

fatal: destination path 'geostd' already exists and is not an empty directory.
Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - \ done

# All requested packages already installed.

Cloning into 'Meeko'...
remote: Enumerating objects: 5373, done.
remote: Counting objects: 100% (1475/1475), done.
remote: Compressing objects: 100% (385/385), done.
remote: Total 5373 (delta 1193), reused 1113 (delta 1090), pack-reused 3898 (from 2)
Receiving objects: 100% (5373/5373), 7.92 MiB | 13.67 MiB/s, done.
Resolving deltas: 100% (3858/3858), done.
Obtaining file:///content/Meeko
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ..

In [ ]:
# @title (2) Install Packages and Data (~ 2min)
%%time

# Get environment configuration from Git Repository
setup_repo="https://github.com/rwxayheee/colab_setup"
!git clone {setup_repo}

# Run setup script
!chmod +x colab_setup/basic_setup.sh
!bash colab_setup/basic_setup.sh

Cloning into 'colab_setup'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 63 (delta 24), reused 45 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 11.99 KiB | 11.99 MiB/s, done.
Resolving deltas: 100% (24/24), done.

[INFO] Step 1: Installing Python packages using Conda and Pip...

Channels:
 - conda-forge
Platform: linux-64
Solving environment: \ | / - \ done

cctbx-base-2024.8    | 66.8 MB   | :   0% 0/1 [00:00<?, ?it/s]
pillow-11.1.0        | 40.1 MB   | :   0% 0/1 [00:00<?, ?it/s]

librdkit-2024.03.6   | 20.1 MB   | :   0% 0/1 [00:00<?, ?it/s]


scipy-1.15.2         | 16.4 MB   | :   0% 0/1 [00:00<?, ?it/s]



pandas-2.2.3         | 15.0 MB   | :   0% 0/1 [00:00<?, ?it/s]




prody-2.4.1          | 13.8 MB   | :   0% 0/1 [00:00<?, ?it/s]





icu-75.1             | 11.6 MB   | :   0% 0/1 [00:00<?, ?it/s]






matplotlib-base-3.10 | 8.1 MB    | :   0

In [ ]:
# @title (3) Import Modules & Locate Command Line Scripts (< 1s)
%%time

# Import modules
import sys, platform
from prody import *
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem
import rdkit, py3Dmol
print("rdkit version:", rdkit.__version__)
print("py3Dmol version:", py3Dmol.__version__)
from ipywidgets import interact, IntSlider
import ipywidgets, copy
from IPython.display import display, Markdown


# Helper functions
def locate_file(from_path = None, query_path = None, query_name = "query file"):

    if not from_path or not query_path:
        raise ValueError("Must specify from_path and query_path")

    possible_path = list(from_path.glob(query_path))

    if not possible_path:
        raise FileNotFoundError(f"Cannot find {query_name} from {from_path} by {query_path}")

    return_which = (
        f"using {query_name} at:\n"
        f"{possible_path[0]}\n"
    )
    print(return_which)

    return possible_path[0]


# Commandline scripts
scrub = locate_file(from_path = Path("/usr/local/bin"), query_path = "scrub.py", query_name = "scrub.py")
mk_prepare_ligand = locate_file(from_path = Path("/usr/local/bin"), query_path = "mk_prepare_ligand.py", query_name = "mk_prepare_ligand.py")
mk_prepare_receptor = locate_file(from_path = Path("/usr/local/bin"), query_path = "mk_prepare_receptor.py", query_name = "mk_prepare_receptor.py")
mk_export = locate_file(from_path = Path("/usr/local/bin"), query_path = "mk_export.py", query_name = "mk_export.py")


# Locate reduce2 in conda install prefix
full_py_version = platform.python_version()
major_and_minor = ".".join(full_py_version.split(".")[:2])
env_path = Path("/usr/local") # default conda install prefix on Colab
reduce2_path = f"lib/python{major_and_minor}/site-packages/mmtbx/command_line/reduce2.py"
reduce2 = locate_file(from_path = env_path, query_path = reduce2_path, query_name = "reduce2.py")


# Locate geostd in current path
geostd_path = locate_file(from_path = Path.cwd(), query_path = "geostd", query_name = "geostd")

#@markdown > Version of imported modules and the location of command line scripts will be reported to output.
#@markdown >
#@markdown > Make sure there are no errors and proceed to the next code block!


rdkit version: 2024.03.6
py3Dmol version: 2.4.2
using scrub.py at:
/usr/local/bin/scrub.py

using mk_prepare_ligand.py at:
/usr/local/bin/mk_prepare_ligand.py

using mk_prepare_receptor.py at:
/usr/local/bin/mk_prepare_receptor.py

using mk_export.py at:
/usr/local/bin/mk_export.py

using reduce2.py at:
/usr/local/lib/python3.11/site-packages/mmtbx/command_line/reduce2.py

using geostd at:
/content/geostd

CPU times: user 1.09 ms, sys: 13 µs, total: 1.1 ms
Wall time: 900 µs


In [ ]:
'''
!obabel PNM_ideal.sdf -O PNM_ideal.pdbqt
!obabel best_design0.pdb -O best_design0.pdbqt -xr
!obabel PNM_ideal.sdf -O PNM_ideal.pdb
'''
!obabel {sdf_protein} -O PNM_ideal.pdbqt
!obabel {new_protein} -O best_design0.pdbqt -xr
!obabel {sdf_protein} -O PNM_ideal.pdb


Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ | done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - openbabel


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    openbabel-3.1.1            |  py311h8b422cb_9         5.0 MB  conda-forge
    ------------------------------------------------------------
                                           Total:         5.0 MB

The following NEW packages will be INSTALLED:

  openbabel          conda-forge/linux-64::openbabel-3.1.1-py311h8b422cb_9 



                                                                        
Preparing transaction: - done
Verifying transaction: | done
Executing transaction: - done
1 molecule converted
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is best_design0.pdb)

1 molecul

# Docking Calculation

In these steps, the exmaple docking calculation is demonstrated in a customizable setup.

*Major customizable variables*

- Ligand Smiles string: `ligand_Smiles`
- Designated pH for ligand preparation: `pH`
- Receptor PDB ID: `pdb_token`
- ProDy selection for receptor atoms: `receptor_selection`
- ProDy selection for box center calculation: `ligand_selection`
- Size of box: `size_x`, `size_y`, `size_z`
- Exhaustiveness of docking: `exhaustiveness`

In [ ]:
'''
!git clone https://github.com/sarisabban/Rg.git
val = !python3 Rg/Rg.py best_design0.pdb
val = val.n
val = float(val.split('=')[1].strip())*2.857
val
'''

In [ ]:
from Bio.PDB import PDBParser
import numpy as np

def calculate_radius_of_gyration_biopython(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)

    atoms = list(structure.get_atoms())
    masses = []
    positions = []

    # Dictionary of approximate atomic masses (in atomic mass units)
    atomic_masses = {
        'H': 1.008, 'C': 12.011, 'N': 14.007, 'O': 15.999, 'P': 30.974, 'S': 32.06
    }

    # Extract masses and positions
    for atom in atoms:
        element = atom.element
        if element in atomic_masses:
            masses.append(atomic_masses[element])
            positions.append(atom.coord)

    masses = np.array(masses)
    positions = np.array(positions)

    # Total mass of the protein
    total_mass = np.sum(masses)

    # Calculate the center of mass
    center_of_mass = np.sum(masses[:, np.newaxis] * positions, axis=0) / total_mass

    # Calculate squared distances of each atom from the center of mass
    squared_distances = np.sum((positions - center_of_mass)**2, axis=1)

    # Calculate the radius of gyration
    radius_of_gyration = np.sqrt(np.sum(masses * squared_distances) / total_mass)

    return radius_of_gyration

# Example usage
pdb_file = "best_design0.pdb" #PNM_ideal.pdb"  # Replace with your file path (works with PDBQT too)
val = calculate_radius_of_gyration_biopython(pdb_file)*2.857
print(f"Radius of gyration: {val:.3f} Å")


Radius of gyration: 45.235 Å


In [ ]:
def calculate_center(pdbqt_file):
    x_sum = y_sum = z_sum = 0.0
    atom_count = 0

    with open(pdbqt_file, 'r') as file:
        for line in file:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                x_sum += x
                y_sum += y
                z_sum += z
                atom_count += 1

    center_x = x_sum / atom_count
    center_y = y_sum / atom_count
    center_z = z_sum / atom_count
    return center_x, center_y, center_z

# Example usage
protein_center = calculate_center("best_design0.pdbqt")
print(f"Center Coordinates: {protein_center}")


Center Coordinates: (-1.5116697177726934, -0.17422654462242626, 1.4751479786422537)


In [ ]:
file = 'config.txt'
with open(file, "w") as file:
  file.write(f"""center_x = {protein_center[0]}
center_y = {protein_center[1]}
center_z = {protein_center[2]}
size_x = {val}
size_y = {val}
size_z = {val}""")

In [ ]:
ligand_SDF = 'PNM_ideal.sdf'
ligandPDBQT = 'PNM_ideal.pdbqt'
#mk_prepare_ligand = "/usr/local/bin/mk_prepare_ligand.py"
#mk_prepare_ligand = "mk_prepare_ligand.py"
!python {mk_prepare_ligand} -i {ligandSDF} -o {ligandPDBQT}

python: can't open file '/content/{mk_prepare_ligand}': [Errno 2] No such file or directory


In [ ]:
prepare_output = "receptor_box"
prepare_inPDB = "best_design0.pdb"
mk_prepare_receptor = "/usr/local/bin/mk_prepare_receptor.py"
!python {mk_prepare_receptor} -i {prepare_inPDB} -o {prepare_output} -p -v --box_center {protein_center[0]} {protein_center[1]} {protein_center[2]} --box_size {val} {val} {val}


@> 1311 atoms and 1 coordinate set(s) were parsed in 0.01s.
No template matched for residue_key='A:119'
tried 3 templates for residue_key='A:119'excess_H_ok=False
GLY        heavy_miss=0 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={1}
NGLY       heavy_miss=0 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={1, 4}
CGLY       heavy_miss=1 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={0, 1}

No template matched for residue_key='A:127'
tried 3 templates for residue_key='A:127'excess_H_ok=False
ARG        heavy_miss=0 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={18}
NARG       heavy_miss=0 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={20, 4}
CARG       heavy_miss=1 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={1, 19}

No template matched for residue_key='A:135'
tried 6 templates for residue_key='A:135'excess_H_ok=False
GLU        heavy_miss=0 heavy_excess=0 H_excess=[] bond_miss=set() bond_excess={13}
NGLU       heavy_miss=0 heavy_ex

In [ ]:
# @title # 2. Docking with Vina Scoring Function (~ 3min)
%%time
#@markdown In this step, the docking calculation is executed by ***Vina***.

#@markdown > Specify the names of inputs. You may include any additional options in `configTXT`.
receptorPDBQT = "best_design0.pdbqt" #@param {type:"string"}
ligandPDBQT = "PNM_ideal.pdbqt" #@param {type:"string"}
configTXT = "config.txt" #@param {type:"string"}
#@markdown > Specify the exhaustiveness of docking. In this example, we use a relatively low `exhaustiveness = 8` for demonstration purposes.
exhaustiveness = 8 #@param {type:"raw"}
#@markdown > A name for the docking output PDBQT file is required.
outputPDBQT = "output.pdbqt" #@param {type:"string"}
outputTXT = "affinity_results.txt"

! ./vina --receptor {receptorPDBQT} --ligand {ligandPDBQT} --config {configTXT} \
       --exhaustiveness {exhaustiveness} \
       --out {outputPDBQT} | tee {outputTXT}

AutoDock Vina v1.2.5
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoDock-V

In [ ]:
# @title # 3. Export and Visualize Docked Poses (~ 1s)
%%time
#@markdown In this step, the docking output is converted to atomistic SDF by **mk_export.py**.

# Export Docked Poses
#@markdown > A name for the result SDF file is required.
dock_outSDF = "1iep_STI_vina_out.sdf" #@param {type:"string"}
! python {mk_export} {outputPDBQT} -s {dock_outSDF}

#@markdown > Finally (and optionally) for visualization:
#@markdown >
#@markdown > The ***Py3DMol*** view will include object from the following files and specs:
# Previously Generated Receptor Files
receptorPDB = "1iep_receptorFH.pdb" #@param {type:"string"}
prepare_output = 'Receptor'
prody_ligandPDB = 'PNM_ideal.pdb'
boxPDB = "1iep_receptorFH.box.pdb" #@param {type:"string"}


refligPDB = 'LIG.pdb' #@param {type:"string"}
reflig_resn = 'STI' #@param {type:"string"}

def Receptor3DView(receptorPDB = None, boxPDB = None, ligPDB = None):

    view = py3Dmol.view()
    view.setBackgroundColor('white')

    view.addModel(open(boxPDB, 'r').read(),'pdb')
    view.addStyle({'stick': {}})
    view.zoomTo()

    view.addModel(open(receptorPDB, 'r').read(),'pdb')
    view.addStyle({'cartoon': {'color':'spectrum', 'opacity': 0.5}})

    if ligPDB is not None:
      view.addModel(open(ligPDB, 'r').read(), 'pdb')
      view.addStyle({'hetflag': True}, {'stick': {}})

    return view

Receptor3DView(receptorPDB = new_protein, \
               boxPDB = prepare_output+'.box.pdb', \
               ligPDB = prody_ligandPDB).show()
'''
# Visualize Docked Poses
def Complex3DView(view, ligmol = None, refligPDB = None, reflig_resn = None):

    new_viewer = copy.deepcopy(view)

    mblock = Chem.MolToMolBlock(ligmol)
    new_viewer.addModel(mblock, 'mol')
    new_viewer.addStyle({'hetflag': True}, {"stick": {'colorscheme': 'greenCarbon'}})

    if refligPDB is not None and reflig_resn is not None:
      new_viewer.addModel(open(refligPDB, 'r').read(), 'pdb')
      new_viewer.addStyle({'resn': reflig_resn}, {"stick": {'colorscheme': 'magentaCarbon', 'opacity': 0.8}})

    return new_viewer


confs = Chem.SDMolSupplier(dock_outSDF)

def conf_viewer(idx):
    mol = confs[idx]
    return Complex3DView(Receptor3DView(receptorPDB = receptorPDB, boxPDB = boxPDB), \
                         mol, \
                         refligPDB = refligPDB, reflig_resn = reflig_resn).show()


interact(conf_viewer, idx=ipywidgets.IntSlider(min=0, max=len(confs)-1, step=1))
'''

/content/Meeko/meeko/cli/mk_export.py:110: UserWarning: molecule 0 not converted to RDKit/SD File
  warnings.warn("molecule %d not converted to RDKit/SD File" % i)
/content/Meeko/meeko/cli/mk_export.py:112: UserWarning: sdf_string does not contain molecular data.
  warnings.warn("sdf_string does not contain molecular data.")
Output SDF will not be created because there is no pose data for ligand. 
Maybe the input poses only contain flexible sidechains and 
keep_flexres_sdf is set to False. 
Use -k with mk_export.py to retain the flexres and write to output SDF File. 


FileNotFoundError: [Errno 2] No such file or directory: 'Receptor.box.pdb'